In [9]:
%%capture
!pip install torch-geometric numpy pandas pyarrow networkx matplotlib seaborn scikit-learn --quiet

In [10]:
from __future__ import annotations

import math
import os
import random
import warnings
from collections import defaultdict
from dataclasses import dataclass, field
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, f1_score

pd.set_option("display.max_columns", 50)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
warnings.filterwarnings("ignore")

# Type alias for Python 3.12
type NodeSet = set[int | str]

@dataclass
class Config:
    # Dataset Paths
    ELLIPTIC_DIR: str = "/kaggle/input/datasets/organizations/ellipticco/elliptic-data-set/elliptic_bitcoin_dataset"
    AMLSIM_DIR: str = "/kaggle/input/datasets/anshankul/ibm-amlsim-example-dataset"
    PAYSIM_DIR: str = "/kaggle/input/datasets/ealaxi/paysim1"
    
    @property
    def ELLIPTIC_FEATURES(self) -> str: return f"{self.ELLIPTIC_DIR}/elliptic_txs_features.csv"
    @property
    def ELLIPTIC_CLASSES(self) -> str: return f"{self.ELLIPTIC_DIR}/elliptic_txs_classes.csv"
    @property
    def ELLIPTIC_EDGES(self) -> str: return f"{self.ELLIPTIC_DIR}/elliptic_txs_edgelist.csv"
    @property
    def AMLSIM_TRANSACTIONS(self) -> str: return f"{self.AMLSIM_DIR}/transactions.csv"
    @property
    def PAYSIM_CSV(self) -> str: return f"{self.PAYSIM_DIR}/PS_20174392719_1491204439457_log.csv"

    # Training Hyperparameters
    RANDOM_SEED: int = 42
    EPOCHS: int = 100
    LR: float = 0.005
    WEIGHT_DECAY: float = 5e-4
    HIDDEN_DIM: int = 64

cfg = Config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Apply deterministic seeds
torch.manual_seed(cfg.RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.RANDOM_SEED)
np.random.seed(cfg.RANDOM_SEED)
random.seed(cfg.RANDOM_SEED)

WORK_DIR = Path(".")
PROCESSED_DIR = WORK_DIR / "processed"
SPLIT_DIR = WORK_DIR / "splits"
PROCESSED_DIR.mkdir(exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Hardware initialization complete. Using device: {device}")

Hardware initialization complete. Using device: cuda


In [11]:
class EllipticLoader:
    __slots__ = ("cfg", "features_df", "classes_df", "edges_df", "merged_df", "graph")

    def __init__(self, cfg: Config) -> None:
        self.cfg = cfg
        self.features_df: pd.DataFrame | None = None
        self.classes_df:  pd.DataFrame | None = None
        self.edges_df:    pd.DataFrame | None = None
        self.merged_df:   pd.DataFrame | None = None
        self.graph:       nx.DiGraph | None   = None

    def load_raw(self) -> "EllipticLoader":
        self.features_df = pd.read_csv(self.cfg.ELLIPTIC_FEATURES, header=None)
        n_feat = self.features_df.shape[1] - 2
        self.features_df.columns = ["txId", "time_step"] + [f"feat_{i}" for i in range(n_feat)]
        self.classes_df = pd.read_csv(self.cfg.ELLIPTIC_CLASSES)
        self.classes_df.columns = ["txId", "class"]
        self.edges_df = pd.read_csv(self.cfg.ELLIPTIC_EDGES)
        self.edges_df.columns = ["txId1", "txId2"]
        return self

    def merge(self) -> "EllipticLoader":
        self.merged_df = self.features_df.merge(self.classes_df, on="txId", how="left")
        def map_label(cls: str) -> int:
            match str(cls):
                case "1": return 1
                case "2": return 0
                case _:   return -1
        self.merged_df["label"] = self.merged_df["class"].apply(map_label)
        return self

class AMLSimLoader:
    COLUMN_ALIASES: dict[str, list[str]] = {
        "tx_id":         ["TX_ID", "Transaction ID"],
        "timestamp":     ["TIMESTAMP", "Timestamp", "Time"],
        "sender":        ["SENDER_ACCOUNT_ID", "Sender Account", "nameOrig"],
        "receiver":      ["RECEIVER_ACCOUNT_ID", "Receiver Account", "nameDest"],
        "tx_type":       ["TX_TYPE", "Type"],
        "amount":        ["TX_AMOUNT", "Amount Received", "Amount"],
        "is_laundering": ["IS_FRAUD", "Is Laundering", "isFraud"],
        "alert_id":      ["ALERT_ID"],
    }

    def __init__(self, cfg: Config) -> None:
        self.cfg = cfg
        self.raw_df: pd.DataFrame | None = None
        self.df:     pd.DataFrame | None = None

    def _resolve_columns(self, df: pd.DataFrame) -> dict[str, str]:
        return {std: alias for std, aliases in self.COLUMN_ALIASES.items() for alias in aliases if alias in df.columns}

    def load_raw(self, nrows: int | None = None) -> "AMLSimLoader":
        self.raw_df = pd.read_csv(self.cfg.AMLSIM_TRANSACTIONS, nrows=nrows)
        return self

    def standardize(self) -> "AMLSimLoader":
        col_map = self._resolve_columns(self.raw_df)
        self.df = self.raw_df.rename(columns={v: k for k, v in col_map.items()})
        keep = [c for c in self.COLUMN_ALIASES if c in self.df.columns]
        self.df = self.df[keep].copy()
        if "is_laundering" in self.df.columns:
            self.df["is_laundering"] = self.df["is_laundering"].astype(int)
        if "timestamp" in self.df.columns:
            self.df["timestamp"] = self.df["timestamp"].astype(int)
        return self

class PaySimLoader:
    __slots__ = ("cfg", "df")
    RENAME_MAP: dict[str, str] = {
        "step":           "timestamp",
        "nameOrig":       "sender",
        "nameDest":       "receiver",
        "amount":         "amount",
        "isFraud":        "is_fraud",
        "isFlaggedFraud": "is_flagged",
        "type":           "tx_type",
    }

    def __init__(self, cfg: Config) -> None:
        self.cfg = cfg
        self.df: pd.DataFrame | None = None

    def load(self, nrows: int | None = None) -> "PaySimLoader":
        self.df = pd.read_csv(self.cfg.PAYSIM_CSV, nrows=nrows)
        self.df = self.df.rename(columns=self.RENAME_MAP)
        self.df["is_fraud"] = self.df["is_fraud"].astype(int)
        self.df["is_flagged"] = self.df["is_flagged"].astype(int)
        return self

In [12]:
def prepare_pyg_data(merged_df, edges_df, tx_id_col, sender_col, receiver_col, label_col, time_col, feature_cols, route_through_tx=False):
    """
    Converts preprocessed DataFrames into a PyTorch Geometric Data object
    with built-in temporal train/val/test split masks.

    route_through_tx=False (Elliptic): edges_df already connects two tx_id-space
        nodes directly via sender_col/receiver_col.
    route_through_tx=True (AMLSim/PaySim): edges_df has one row per transaction
        with tx_id_col/sender_col/receiver_col as three separate columns on the
        SAME row (account IDs live in a different namespace from tx_id). Builds
        sender -> tx_id -> receiver edges internally so the labeled transaction
        node has real edges, instead of leaving it isolated while sender/receiver
        accounts connect directly to each other.
    """
    unique_nodes = pd.concat([merged_df[tx_id_col], edges_df[sender_col], edges_df[receiver_col]]).unique()
    node_to_idx = {node: idx for idx, node in enumerate(unique_nodes)}

    num_nodes = len(unique_nodes)
    num_features = len(feature_cols)

    x_map = np.zeros((num_nodes, num_features), dtype=np.float32)
    feature_map = merged_df.set_index(tx_id_col)[feature_cols]
    for node, idx in node_to_idx.items():
        if node in feature_map.index:
            x_map[idx] = feature_map.loc[node].values

    x = torch.tensor(x_map, dtype=torch.float32)

    if route_through_tx:
        tx_idx = edges_df[tx_id_col].map(node_to_idx).values
        sender_idx = edges_df[sender_col].map(node_to_idx).values
        receiver_idx = edges_df[receiver_col].map(node_to_idx).values
        edge_src = np.concatenate([sender_idx, tx_idx])
        edge_dst = np.concatenate([tx_idx, receiver_idx])
    else:
        edge_src = edges_df[sender_col].map(node_to_idx).values
        edge_dst = edges_df[receiver_col].map(node_to_idx).values

    edge_index = torch.tensor(np.array([edge_src, edge_dst]), dtype=torch.long)

    y_map = np.full(num_nodes, -1, dtype=np.int64)
    label_map = merged_df.set_index(tx_id_col)[label_col]
    for node, idx in node_to_idx.items():
        if node in label_map.index:
            y_map[idx] = label_map.loc[node]

    y = torch.tensor(y_map, dtype=torch.long)

    time_map = merged_df.set_index(tx_id_col)[time_col]
    node_times = np.zeros(num_nodes, dtype=np.int32)
    for node, idx in node_to_idx.items():
        if node in time_map.index:
            node_times[idx] = time_map.loc[node]

    max_time = node_times.max()
    train_thresh = int(max_time * 0.60)
    val_thresh = int(max_time * 0.80)

    labeled_mask = y_map != -1

    train_mask = torch.tensor((node_times <= train_thresh) & labeled_mask, dtype=torch.bool)
    val_mask = torch.tensor((node_times > train_thresh) & (node_times <= val_thresh) & labeled_mask, dtype=torch.bool)
    test_mask = torch.tensor((node_times > val_thresh) & labeled_mask, dtype=torch.bool)

    time_step = torch.tensor(node_times, dtype=torch.long)
    return Data(x=x, edge_index=edge_index, y=y, time_step=time_step,
                train_mask=train_mask, val_mask=val_mask, test_mask=test_mask)

In [13]:
class DACISGraphSAGE(nn.Module):
    def __init__(self, in_channels: int, hidden_channels: int, out_channels: int, dropout: float = 0.3):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels, aggr='max')
        self.bn1 = nn.BatchNorm1d(hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels, aggr='max')
        self.bn2 = nn.BatchNorm1d(hidden_channels)
        self.linear = nn.Linear(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        return self.linear(x)

In [14]:
def compute_metrics(logits: torch.Tensor, labels: torch.Tensor, mask: torch.Tensor, threshold: float = 0.5) -> dict:
    with torch.no_grad():
        mask_idx = mask.nonzero(as_tuple=False).view(-1)
        if mask_idx.numel() == 0:
            return {"auroc": 0.0, "auprc": 0.0, "f1": 0.0, "probs": np.array([]), "labels": np.array([])}

        filtered_logits = logits[mask_idx]
        filtered_labels = labels[mask_idx].cpu().numpy()

        probs = F.softmax(filtered_logits, dim=1)[:, 1].cpu().numpy()
        preds = (probs >= threshold).astype(int)

        if len(np.unique(filtered_labels)) < 2:
            return {"auroc": 0.5, "auprc": 0.0, "f1": 0.0, "probs": probs, "labels": filtered_labels}

        auroc = roc_auc_score(filtered_labels, probs)
        precision, recall, _ = precision_recall_curve(filtered_labels, probs)
        auprc = auc(recall, precision)
        f1 = f1_score(filtered_labels, preds, zero_division=0)

        return {"auroc": auroc, "auprc": auprc, "f1": f1, "probs": probs, "labels": filtered_labels}


def best_threshold_for_f1(probs: np.ndarray, labels: np.ndarray) -> float:
    if len(np.unique(labels)) < 2:
        return 0.5
    precision, recall, thresholds = precision_recall_curve(labels, probs)
    f1s = 2 * precision * recall / (precision + recall + 1e-12)
    best_idx = np.nanargmax(f1s[:-1])
    return float(thresholds[best_idx])


def train_and_evaluate(dataset_name: str, data, epochs=100, lr=0.005, weight_decay=5e-4, hidden_dim=64) -> None:
    data = data.to(device)

    train_labels = data.y[data.train_mask].cpu().numpy()
    num_neg = np.sum(train_labels == 0)
    num_pos = np.sum(train_labels == 1)
    pos_weight = float(num_neg) / max(num_pos, 1)

    criterion = nn.CrossEntropyLoss(weight=torch.tensor([1.0, pos_weight * 1.5], dtype=torch.float32).to(device))

    model = DACISGraphSAGE(
        in_channels=data.num_node_features,
        hidden_channels=hidden_dim,
        out_channels=2
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_score = -1.0
    best_weights_path = WORK_DIR / f"{dataset_name.lower()}_best_weights.pt"

    print(f"===========================================================")
    print(f" Executing Optimization Pipeline: {dataset_name} ")
    print(f" Calculated Anomaly Loss Scaling Multiplier: {pos_weight:.2f}")
    print(f"===========================================================")

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)

        loss = criterion(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_out = model(data.x, data.edge_index)
            val_metrics = compute_metrics(val_out, data.y, data.val_mask)

        # FIX: select checkpoint on AUPRC alone — F1@0.5 is near-zero/noisy
        # under heavy imbalance and was destabilizing model selection.
        current_val_score = val_metrics["auprc"]

        if current_val_score > best_val_score:
            best_val_score = current_val_score
            torch.save(model.state_dict(), best_weights_path)

        if epoch % 10 == 0 or epoch == 1:
            print(f"Epoch {epoch:03d} | Train Loss: {loss.item():.4f} | Val AUROC: {val_metrics['auroc']:.4f} | Val AUPRC: {val_metrics['auprc']:.4f} | Val F1: {val_metrics['f1']:.4f}")

    print(f"\nEvaluating peak validation parameters against test track...")
    model.load_state_dict(torch.load(best_weights_path, weights_only=True))
    model.eval()

    with torch.no_grad():
        # FIX: tune the decision threshold on val, then apply it to test —
        # corrects the F1 collapse caused by the hard argmax/0.5 threshold.
        val_out = model(data.x, data.edge_index)
        val_final_metrics = compute_metrics(val_out, data.y, data.val_mask)
        best_thresh = best_threshold_for_f1(val_final_metrics["probs"], val_final_metrics["labels"])

        final_out = model(data.x, data.edge_index)
        test_metrics = compute_metrics(final_out, data.y, data.test_mask, threshold=best_thresh)

    print(f"-----------------------------------------------------------")
    print(f" {dataset_name.upper()} FINAL TEST RESULTS")
    print(f"-----------------------------------------------------------")
    print(f"Selected Threshold : {best_thresh:.4f}")
    print(f"AUROC : {test_metrics['auroc']:.4f}")
    print(f"AUPRC : {test_metrics['auprc']:.4f}")
    print(f"F1    : {test_metrics['f1']:.4f}")
    print(f"Saved : {best_weights_path.resolve()}\n")

In [15]:
if os.path.exists(cfg.ELLIPTIC_DIR):
    elliptic = EllipticLoader(cfg).load_raw().merge()
    feature_cols = [col for col in elliptic.merged_df.columns if col.startswith('feat_')]
    
    elliptic_pyg = prepare_pyg_data(
        merged_df=elliptic.merged_df, edges_df=elliptic.edges_df,
        tx_id_col='txId', sender_col='txId1', receiver_col='txId2',
        label_col='label', time_col='time_step', feature_cols=feature_cols,
        route_through_tx=False,
    )
    train_and_evaluate("Elliptic", elliptic_pyg, epochs=cfg.EPOCHS, lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY, hidden_dim=cfg.HIDDEN_DIM)
else:
    print(f"Skipping Elliptic Dataset: Directory not found at {cfg.ELLIPTIC_DIR}")

 Executing Optimization Pipeline: Elliptic 
 Calculated Anomaly Loss Scaling Multiplier: 8.19
Epoch 001 | Train Loss: 0.7123 | Val AUROC: 0.7563 | Val AUPRC: 0.2636 | Val F1: 0.3583
Epoch 010 | Train Loss: 0.2962 | Val AUROC: 0.9204 | Val AUPRC: 0.6572 | Val F1: 0.4335
Epoch 020 | Train Loss: 0.2075 | Val AUROC: 0.9522 | Val AUPRC: 0.7688 | Val F1: 0.4053
Epoch 030 | Train Loss: 0.1645 | Val AUROC: 0.9514 | Val AUPRC: 0.7723 | Val F1: 0.3852
Epoch 040 | Train Loss: 0.1363 | Val AUROC: 0.9493 | Val AUPRC: 0.7474 | Val F1: 0.3794
Epoch 050 | Train Loss: 0.1143 | Val AUROC: 0.9319 | Val AUPRC: 0.6006 | Val F1: 0.3794
Epoch 060 | Train Loss: 0.0982 | Val AUROC: 0.9196 | Val AUPRC: 0.4950 | Val F1: 0.3908
Epoch 070 | Train Loss: 0.0817 | Val AUROC: 0.9420 | Val AUPRC: 0.6439 | Val F1: 0.4117
Epoch 080 | Train Loss: 0.0731 | Val AUROC: 0.9295 | Val AUPRC: 0.5559 | Val F1: 0.4115
Epoch 090 | Train Loss: 0.0688 | Val AUROC: 0.9588 | Val AUPRC: 0.7564 | Val F1: 0.4460
Epoch 100 | Train Loss: 0.

In [16]:
if os.path.exists(cfg.AMLSIM_DIR):
    amlsim = AMLSimLoader(cfg).load_raw(nrows=None).standardize()

    aml_df = amlsim.df.copy()

    aml_df['feat_amount'] = np.log1p(aml_df['amount'])
    aml_df['feat_amount'] = (aml_df['feat_amount'] - aml_df['feat_amount'].mean()) / (aml_df['feat_amount'].std() + 1e-8)
    aml_df['feat_type'] = aml_df['tx_type'].astype('category').cat.codes

    aml_edges = aml_df[['tx_id', 'sender', 'receiver']].copy()

    amlsim_pyg = prepare_pyg_data(
        merged_df=aml_df,
        edges_df=aml_edges,
        tx_id_col='tx_id',
        sender_col='sender',
        receiver_col='receiver',
        label_col='is_laundering',
        time_col='timestamp',
        feature_cols=['feat_amount', 'feat_type'],
        route_through_tx=True,
    )
    train_and_evaluate("AMLSim", amlsim_pyg, epochs=cfg.EPOCHS, lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY, hidden_dim=cfg.HIDDEN_DIM)
else:
    print(f"Skipping AMLSim Dataset: Directory not found at {cfg.AMLSIM_DIR}")

 Executing Optimization Pipeline: AMLSim 
 Calculated Anomaly Loss Scaling Multiplier: 782.39
Epoch 001 | Train Loss: 0.7650 | Val AUROC: 0.8956 | Val AUPRC: 0.0825 | Val F1: 0.0058
Epoch 010 | Train Loss: 0.4002 | Val AUROC: 0.9146 | Val AUPRC: 0.3871 | Val F1: 0.0086
Epoch 020 | Train Loss: 0.3234 | Val AUROC: 0.9144 | Val AUPRC: 0.3553 | Val F1: 0.0087
Epoch 030 | Train Loss: 0.3044 | Val AUROC: 0.9170 | Val AUPRC: 0.3740 | Val F1: 0.0088
Epoch 040 | Train Loss: 0.2950 | Val AUROC: 0.9184 | Val AUPRC: 0.4066 | Val F1: 0.0088
Epoch 050 | Train Loss: 0.2908 | Val AUROC: 0.9184 | Val AUPRC: 0.3798 | Val F1: 0.0090
Epoch 060 | Train Loss: 0.2921 | Val AUROC: 0.9179 | Val AUPRC: 0.4254 | Val F1: 0.0091
Epoch 070 | Train Loss: 0.2875 | Val AUROC: 0.9182 | Val AUPRC: 0.4101 | Val F1: 0.0092
Epoch 080 | Train Loss: 0.2887 | Val AUROC: 0.9175 | Val AUPRC: 0.4300 | Val F1: 0.0094
Epoch 090 | Train Loss: 0.2879 | Val AUROC: 0.9180 | Val AUPRC: 0.4200 | Val F1: 0.0096
Epoch 100 | Train Loss: 0.

In [17]:
if os.path.exists(cfg.PAYSIM_DIR):
    paysim = PaySimLoader(cfg).load(nrows=500000)

    pay_df = paysim.df.copy()
    pay_df['tx_id'] = pay_df.index

    pay_df['feat_amount'] = np.log1p(pay_df['amount'])
    pay_df['feat_amount'] = (pay_df['feat_amount'] - pay_df['feat_amount'].mean()) / (pay_df['feat_amount'].std() + 1e-8)
    pay_df['feat_type'] = pay_df['tx_type'].astype('category').cat.codes

    pay_edges = pay_df[['tx_id', 'sender', 'receiver']].copy()

    paysim_pyg = prepare_pyg_data(
        merged_df=pay_df,
        edges_df=pay_edges,
        tx_id_col='tx_id',
        sender_col='sender',
        receiver_col='receiver',
        label_col='is_fraud',
        time_col='timestamp',
        feature_cols=['feat_amount', 'feat_type'],
        route_through_tx=True,
    )
    train_and_evaluate("PaySim", paysim_pyg, epochs=cfg.EPOCHS, lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY, hidden_dim=cfg.HIDDEN_DIM)
else:
    print(f"Skipping PaySim Dataset: Directory not found at {cfg.PAYSIM_DIR}")

 Executing Optimization Pipeline: PaySim 
 Calculated Anomaly Loss Scaling Multiplier: 1290.57
Epoch 001 | Train Loss: 0.6709 | Val AUROC: 0.7718 | Val AUPRC: 0.0354 | Val F1: 0.0007
Epoch 010 | Train Loss: 0.5818 | Val AUROC: 0.7931 | Val AUPRC: 0.0094 | Val F1: 0.0010
Epoch 020 | Train Loss: 0.5255 | Val AUROC: 0.8270 | Val AUPRC: 0.0122 | Val F1: 0.0015
Epoch 030 | Train Loss: 0.4752 | Val AUROC: 0.8591 | Val AUPRC: 0.0155 | Val F1: 0.0024
Epoch 040 | Train Loss: 0.4491 | Val AUROC: 0.8906 | Val AUPRC: 0.0927 | Val F1: 0.0028
Epoch 050 | Train Loss: 0.4217 | Val AUROC: 0.9040 | Val AUPRC: 0.0566 | Val F1: 0.0024
Epoch 060 | Train Loss: 0.3785 | Val AUROC: 0.9031 | Val AUPRC: 0.0384 | Val F1: 0.0021
Epoch 070 | Train Loss: 0.3869 | Val AUROC: 0.9046 | Val AUPRC: 0.0208 | Val F1: 0.0022
Epoch 080 | Train Loss: 0.3619 | Val AUROC: 0.9063 | Val AUPRC: 0.0212 | Val F1: 0.0029
Epoch 090 | Train Loss: 0.3690 | Val AUROC: 0.9052 | Val AUPRC: 0.0391 | Val F1: 0.0025
Epoch 100 | Train Loss: 0

In [18]:
print("=======================================================")
print(" EXPORTING PROCESSED PYG GRAPHS FOR INFERENCE NOTEBOOK ")
print("=======================================================")

if 'elliptic_pyg' in globals():
    elliptic_path = PROCESSED_DIR / "elliptic_pyg_data.pt"
    torch.save(elliptic_pyg.to_dict(), elliptic_path)
    print(f"Saved Elliptic graph data to: {elliptic_path}")

if 'amlsim_pyg' in globals():
    amlsim_path = PROCESSED_DIR / "amlsim_pyg_data.pt"
    torch.save(amlsim_pyg.to_dict(), amlsim_path)
    print(f"Saved AMLSim graph data to: {amlsim_path}")

if 'paysim_pyg' in globals():
    paysim_path = PROCESSED_DIR / "paysim_pyg_data.pt"
    torch.save(paysim_pyg.to_dict(), paysim_path)
    print(f"Saved PaySim graph data to: {paysim_path}")

print("=======================================================")
print(" Pipeline Execution Complete ")
print("=======================================================")

 EXPORTING PROCESSED PYG GRAPHS FOR INFERENCE NOTEBOOK 
Saved Elliptic graph data to: processed/elliptic_pyg_data.pt
Saved AMLSim graph data to: processed/amlsim_pyg_data.pt
Saved PaySim graph data to: processed/paysim_pyg_data.pt
 Pipeline Execution Complete 
